# F1 Race Analytics — 03: SQL Analysis
**Approach:** Load processed CSVs into an in-memory SQLite database, then run analytical queries  
**SQL features used:** Window functions (RANK, ROW_NUMBER, LAG), CTEs, aggregations, CASE statements  
**Questions answered:** 10 business-style analytical questions

---

In [1]:
import sqlite3
import pandas as pd

# Load processed data
df_results   = pd.read_csv('../data/processed/race_results_clean.csv')
df_driver    = pd.read_csv('../data/processed/driver_season_stats.csv')
df_pole      = pd.read_csv('../data/processed/pole_conversion.csv')
df_pit       = pd.read_csv('../data/processed/pit_stop_summary.csv')
df_const     = pd.read_csv('../data/raw/constructor_standings.csv')

# Create in-memory SQLite DB and load tables
conn = sqlite3.connect(':memory:')
df_results.to_sql('race_results', conn, index=False, if_exists='replace')
df_driver.to_sql('driver_stats',  conn, index=False, if_exists='replace')
df_pole.to_sql('pole_conversion', conn, index=False, if_exists='replace')
df_pit.to_sql('pit_stops',        conn, index=False, if_exists='replace')
df_const.to_sql('constructor_standings', conn, index=False, if_exists='replace')

def query(sql, title=''):
    if title:
        print(f'\n{title}')
        print('-' * len(title))
    result = pd.read_sql_query(sql, conn)
    return result

print('SQLite DB ready. Tables loaded: race_results, driver_stats, pole_conversion, pit_stops, constructor_standings')

SQLite DB ready. Tables loaded: race_results, driver_stats, pole_conversion, pit_stops, constructor_standings


## Q1. Which constructor won the most races each season?

In [ ]:
query("""
WITH season_wins AS (
    SELECT
        season,
        constructor,
        COUNT(*) AS wins,
        RANK() OVER (PARTITION BY season ORDER BY COUNT(*) DESC) AS win_rank
    FROM race_results
    WHERE finish_position = 1
    GROUP BY season, constructor
)
SELECT season, constructor, wins
FROM season_wins
WHERE win_rank = 1
ORDER BY season
""", 'Q1: Dominant Constructor per Season')


Q1: Dominant Constructor per Season
-----------------------------------


,season,constructor,wins
0,2018,Mercedes,11
1,2019,Mercedes,15
2,2020,Mercedes,13
3,2021,Red Bull,11
4,2022,Red Bull,17
5,2023,Red Bull,21
6,2024,Red Bull,9


## Q2. Pole-to-win conversion rate — does starting first guarantee winning?

In [6]:
query("""
SELECT
    season,
    COUNT(*) AS total_races,
    SUM(CASE WHEN pole_converted = 1 THEN 1 ELSE 0 END) AS poles_converted,
    ROUND(100.0 * SUM(CASE WHEN pole_converted = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS conversion_pct
FROM pole_conversion
GROUP BY season
ORDER BY season
""", 'Q2: Pole Position → Win Conversion Rate by Season')


Q2: Pole Position → Win Conversion Rate by Season
-------------------------------------------------


,season,total_races,poles_converted,conversion_pct
0,2018,21,10,47.6
1,2019,21,8,38.1
2,2020,17,10,58.8
3,2021,22,12,54.5
4,2022,22,10,45.5
5,2023,22,15,68.2
6,2024,24,12,50.0


## Q3. Top 10 drivers by total points across 2018–2024

In [7]:
query("""
SELECT
    driver_name,
    SUM(total_points) AS career_points,
    SUM(wins)         AS career_wins,
    SUM(podiums)      AS career_podiums,
    SUM(races)        AS career_races,
    ROUND(SUM(total_points) * 1.0 / SUM(races), 1) AS points_per_race
FROM driver_stats
GROUP BY driver_name
HAVING SUM(races) >= 20
ORDER BY career_points DESC
LIMIT 10
""", 'Q3: Top 10 Drivers by Career Points (2018-2024, min 20 races)')


Q3: Top 10 Drivers by Career Points (2018-2024, min 20 races)
-------------------------------------------------------------


,driver_name,career_points,career_wins,career_podiums,career_races,points_per_race
0,Max Verstappen,2491.5,60,101,149,16.7
1,Lewis Hamilton,2210.5,43,85,148,14.9
2,Charles Leclerc,1363.0,8,43,149,9.1
3,Sergio Pérez,1118.0,6,32,147,7.6
4,Carlos Sainz,1085.5,4,27,148,7.3
5,Valtteri Bottas,1072.0,7,45,149,7.2
6,Lando Norris,950.0,4,26,128,7.4
7,Sebastian Vettel,673.0,6,23,101,6.7
8,George Russell,664.0,3,15,128,5.2
9,Daniel Ricciardo,504.0,3,5,128,3.9


## Q4. Which circuits produce the most position changes (most overtaking)?

In [8]:
query("""
SELECT
    circuit,
    country,
    COUNT(DISTINCT season || round) AS race_count,
    ROUND(AVG(ABS(position_delta)), 2) AS avg_positions_moved,
    ROUND(AVG(CASE WHEN position_delta > 0 THEN position_delta END), 2) AS avg_positions_gained
FROM race_results
WHERE position_delta IS NOT NULL
GROUP BY circuit, country
HAVING race_count >= 3
ORDER BY avg_positions_moved DESC
LIMIT 10
""", 'Q4: Circuits with Most Position Changes (overtaking-friendly)')


Q4: Circuits with Most Position Changes (overtaking-friendly)
-------------------------------------------------------------


,circuit,country,race_count,avg_positions_moved,avg_positions_gained
0,Autódromo José Carlos Pace,Brazil,6,4.25,4.54
1,Circuit of the Americas,USA,6,4.21,3.98
2,Baku City Circuit,Azerbaijan,6,4.14,3.73
3,Autodromo Enzo e Dino Ferrari,Italy,4,3.99,3.58
4,Albert Park Grand Prix Circuit,Australia,5,3.96,3.44
5,Autodromo Nazionale di Monza,Italy,7,3.86,3.62
6,Bahrain International Circuit,Bahrain,8,3.86,3.49
7,Hungaroring,Hungary,7,3.79,4.45
8,Sochi Autodrom,Russia,4,3.73,4.67
9,Red Bull Ring,Austria,9,3.67,3.55


## Q5. Constructor points gap between 1st and 2nd place per season

In [9]:
query("""
WITH ranked AS (
    SELECT
        season,
        constructor,
        points,
        ROW_NUMBER() OVER (PARTITION BY season ORDER BY points DESC) AS pos_rank
    FROM constructor_standings
),
top2 AS (
    SELECT season,
        MAX(CASE WHEN pos_rank = 1 THEN constructor END) AS champion,
        MAX(CASE WHEN pos_rank = 1 THEN points END)      AS champion_points,
        MAX(CASE WHEN pos_rank = 2 THEN constructor END) AS runner_up,
        MAX(CASE WHEN pos_rank = 2 THEN points END)      AS runner_up_points
    FROM ranked
    GROUP BY season
)
SELECT
    season,
    champion,
    champion_points,
    runner_up,
    runner_up_points,
    ROUND(champion_points - runner_up_points, 0) AS points_gap
FROM top2
ORDER BY season
""", 'Q5: Championship Dominance — Points Gap Between 1st and 2nd')


Q5: Championship Dominance — Points Gap Between 1st and 2nd
-----------------------------------------------------------


,season,champion,champion_points,runner_up,runner_up_points,points_gap
0,2018,Mercedes,655.0,Ferrari,571.0,84.0
1,2019,Mercedes,739.0,Ferrari,504.0,235.0
2,2020,Mercedes,573.0,Red Bull,319.0,254.0
3,2021,Mercedes,613.5,Red Bull,585.5,28.0
4,2022,Red Bull,759.0,Ferrari,554.0,205.0
5,2023,Red Bull,860.0,Mercedes,409.0,451.0
6,2024,McLaren,666.0,Ferrari,652.0,14.0


## Q6. Driver consistency — best podium rate vs lowest DNF rate

In [10]:
query("""
SELECT
    driver_name,
    SUM(races)   AS total_races,
    SUM(podiums) AS total_podiums,
    SUM(dnfs)    AS total_dnfs,
    ROUND(100.0 * SUM(podiums) / SUM(races), 1) AS podium_rate_pct,
    ROUND(100.0 * SUM(dnfs) / SUM(races), 1)    AS dnf_rate_pct,
    ROUND(
        (1.0 * SUM(podiums) / SUM(races)) - (1.0 * SUM(dnfs) / SUM(races)),
    3) AS consistency_score
FROM driver_stats
GROUP BY driver_name
HAVING total_races >= 20
ORDER BY consistency_score DESC
LIMIT 12
""", 'Q6: Driver Consistency Score (min 20 race starts)')


Q6: Driver Consistency Score (min 20 race starts)
-------------------------------------------------


,driver_name,total_races,total_podiums,total_dnfs,podium_rate_pct,dnf_rate_pct,consistency_score
0,Max Verstappen,149,101,17,67.8,11.4,0.564
1,Lewis Hamilton,148,85,8,57.4,5.4,0.520
2,Valtteri Bottas,149,45,20,30.2,13.4,0.168
3,Oscar Piastri,46,10,3,21.7,6.5,0.152
4,Charles Leclerc,149,43,24,28.9,16.1,0.128
5,Lando Norris,128,26,11,20.3,8.6,0.117
6,Sebastian Vettel,101,23,14,22.8,13.9,0.089
7,Sergio Pérez,147,32,19,21.8,12.9,0.088
8,Carlos Sainz,148,27,21,18.2,14.2,0.041
9,Kimi Räikkönen,79,12,9,15.2,11.4,0.038


## Q7. Season-on-season points improvement per constructor (YoY delta)

In [11]:
query("""
WITH yoy AS (
    SELECT
        season,
        constructor,
        points,
        LAG(points) OVER (PARTITION BY constructor ORDER BY season) AS prev_season_points
    FROM constructor_standings
)
SELECT
    season,
    constructor,
    points,
    prev_season_points,
    ROUND(points - prev_season_points, 0) AS yoy_delta
FROM yoy
WHERE prev_season_points IS NOT NULL
ORDER BY ABS(yoy_delta) DESC
LIMIT 10
""", 'Q7: Biggest Season-on-Season Points Swings (LAG window function)')


Q7: Biggest Season-on-Season Points Swings (LAG window function)
----------------------------------------------------------------


,season,constructor,points,prev_season_points,yoy_delta
0,2020,Ferrari,131.0,504.0,-373.0
1,2024,McLaren,666.0,302.0,364.0
2,2024,Red Bull,589.0,860.0,-271.0
3,2021,Red Bull,585.5,319.0,267.0
4,2024,Ferrari,652.0,406.0,246.0
5,2022,Ferrari,554.0,323.5,231.0
6,2023,Aston Martin,280.0,55.0,225.0
7,2021,Ferrari,323.5,131.0,193.0
8,2024,Aston Martin,94.0,280.0,-186.0
9,2022,Red Bull,759.0,585.5,174.0


## Q8. Which drivers improved the most from grid to finish on average?

In [12]:
query("""
SELECT
    driver_name,
    constructor,
    COUNT(*) AS races,
    ROUND(AVG(position_delta), 2) AS avg_positions_gained,
    ROUND(AVG(CASE WHEN position_delta > 0 THEN position_delta END), 2) AS avg_gain_when_positive,
    MAX(position_delta) AS best_single_race_gain
FROM race_results
WHERE position_delta IS NOT NULL AND season BETWEEN 2018 AND 2024
GROUP BY driver_name, constructor
HAVING races >= 15
ORDER BY avg_positions_gained DESC
LIMIT 10
""", 'Q8: Drivers Who Gain the Most Positions from Grid to Finish')


Q8: Drivers Who Gain the Most Positions from Grid to Finish
-----------------------------------------------------------


,driver_name,constructor,races,avg_positions_gained,avg_gain_when_positive,best_single_race_gain
0,Stoffel Vandoorne,McLaren,21,3.05,4.93,7.0
1,Lance Stroll,Williams,21,2.10,3.86,7.0
2,Guanyu Zhou,Sauber,24,2.08,3.28,6.0
3,Marcus Ericsson,Sauber,21,2.05,4.80,10.0
4,Kimi Räikkönen,Alfa Romeo,54,1.70,3.72,9.0
5,Robert Kubica,Williams,18,1.56,3.09,8.0
6,Daniil Kvyat,Toro Rosso,21,1.52,6.00,12.0
7,Nicholas Latifi,Williams,59,1.44,3.66,11.0
8,Daniil Kvyat,AlphaTauri,17,1.18,3.40,6.0
9,Lance Stroll,Aston Martin,87,1.14,4.57,14.0


## Q9. DNF analysis — which constructors are most unreliable?

In [13]:
query("""
SELECT
    constructor,
    COUNT(*) AS total_entries,
    SUM(CASE WHEN dnf_flag = 1 THEN 1 ELSE 0 END) AS dnfs,
    ROUND(100.0 * SUM(CASE WHEN dnf_flag = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS dnf_rate_pct,
    RANK() OVER (ORDER BY
        100.0 * SUM(CASE WHEN dnf_flag = 1 THEN 1 ELSE 0 END) / COUNT(*) DESC
    ) AS reliability_rank
FROM race_results
GROUP BY constructor
HAVING total_entries >= 30
ORDER BY dnf_rate_pct DESC
LIMIT 10
""", 'Q9: Constructor Reliability — DNF Rate Ranking')


Q9: Constructor Reliability — DNF Rate Ranking
----------------------------------------------


,constructor,total_entries,dnfs,dnf_rate_pct,reliability_rank
0,Williams,297,61,20.5,1
1,Renault,118,24,20.3,2
2,Toro Rosso,84,17,20.2,3
3,Haas F1 Team,298,58,19.5,4
4,Force India,42,8,19.0,5
5,Racing Point,76,13,17.1,6
6,AlphaTauri,166,27,16.3,7
7,Alpine F1 Team,180,29,16.1,8
8,Sauber,90,14,15.6,9
9,RB F1 Team,48,7,14.6,10


## Q10. Fastest average pit stop teams — does speed in the pit lane translate to wins?

In [14]:
query("""
WITH pit_by_constructor AS (
    SELECT
        r.constructor,
        ROUND(AVG(p.avg_stop_duration), 3) AS avg_pit_duration_sec,
        ROUND(MIN(p.fastest_stop), 3)       AS best_ever_stop_sec,
        COUNT(DISTINCT r.season || r.round)  AS races_with_pit_data
    FROM pit_stops p
    JOIN race_results r
      ON p.season = r.season AND p.round = r.round AND p.driver_id = r.driver_id
    WHERE p.avg_stop_duration IS NOT NULL AND p.avg_stop_duration BETWEEN 1.5 AND 60
    GROUP BY r.constructor
    HAVING races_with_pit_data >= 10
),
constructor_wins AS (
    SELECT constructor, SUM(CASE WHEN finish_position = 1 THEN 1 ELSE 0 END) AS wins
    FROM race_results GROUP BY constructor
)
SELECT
    p.constructor,
    p.avg_pit_duration_sec,
    p.best_ever_stop_sec,
    w.wins,
    RANK() OVER (ORDER BY p.avg_pit_duration_sec ASC) AS pit_speed_rank
FROM pit_by_constructor p
LEFT JOIN constructor_wins w ON p.constructor = w.constructor
ORDER BY avg_pit_duration_sec ASC
LIMIT 10
""", 'Q10: Pit Stop Speed vs Race Wins — Does Pit Lane Efficiency Win Races?')



Q10: Pit Stop Speed vs Race Wins — Does Pit Lane Efficiency Win Races?
----------------------------------------------------------------------


,constructor,avg_pit_duration_sec,best_ever_stop_sec,wins,pit_speed_rank
0,Red Bull,24.329,14.113,67,1
1,RB F1 Team,24.356,17.535,0,2
2,Mercedes,24.416,14.155,53,3
3,Ferrari,24.496,14.008,19,4
4,Alpine F1 Team,24.642,14.041,1,5
5,Aston Martin,24.729,14.118,0,6
6,Renault,24.925,17.455,0,7
7,McLaren,24.983,14.493,7,8
8,AlphaTauri,25.071,13.973,1,9
9,Toro Rosso,25.092,17.257,0,10


In [15]:
conn.close()
print('SQL analysis complete. All 10 queries executed.')
print('Key insight: Window functions (RANK, ROW_NUMBER, LAG) used in Q1, Q5, Q7, Q9, Q10.')
print('CTEs used in Q1, Q5, Q7, Q10 for multi-step logic.')

SQL analysis complete. All 10 queries executed.
Key insight: Window functions (RANK, ROW_NUMBER, LAG) used in Q1, Q5, Q7, Q9, Q10.
CTEs used in Q1, Q5, Q7, Q10 for multi-step logic.
